# t-SNE：高维数据的"显微镜"
> 难度标签：中级 | 预计时长：15-25分钟 | 前置知识：无学习经验


> 所属模块：03_无监督学习/降维 | 源文件：t_SNE.py | 核心功能：非线性降维可视化、perplexity 调参、与 PCA 对比

## 概述

t-SNE（t-distributed Stochastic Neighbor Embedding）是目前最流行的高维数据可视化工具。它把高维数据映射到 2D 或 3D 空间，同时**保持数据的局部邻域结构**——高维空间中靠近的点在低维空间中依然靠近。

核心思想：在高维空间中用高斯分布建模点对的相似性，在低维空间中用 t 分布建模，然后最小化两个分布之间的 KL 散度。

脚本在 Iris 和 Digits 数据集上演示了 t-SNE 的降维效果和参数影响。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
from sklearn.datasets import load_iris, load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

## 数学原理

### 1. 高维空间中的概率分布

**代码对应**：`TSNE(n_components=2, perplexity=30)` 执行 t-SNE 降维。

t-SNE 在高维空间中定义样本对之间的**条件概率**：

$$p_{j|i} = \frac{\exp(-\|\mathbf{x}_i - \mathbf{x}_j\|^2 / 2\sigma_i^2)}{\sum_{k \neq i}\exp(-\|\mathbf{x}_i - \mathbf{x}_k\|^2 / 2\sigma_i^2)}$$

对称化后：

$$p_{ij} = \frac{p_{j|i} + p_{i|j}}{2n}$$

$\sigma_i$ 由 **perplexity** 参数确定：

$$\text{Perp}(P_i) = 2^{H(P_i)}, \quad H(P_i) = -\sum_{j}p_{j|i}\log_2 p_{j|i}$$

perplexity 可以理解为有效邻居数。**代码对应**：`perplexity=30` 意味着每个点大约有 30 个"有效邻居"。

### 2. 低维空间中的概率分布

在低维空间（通常 2D）中，使用**Student-t 分布**（自由度为 1，即柯西分布）：

$$q_{ij} = \frac{(1 + \|\mathbf{y}_i - \mathbf{y}_j\|^2)^{-1}}{\sum_{k \neq l}(1 + \|\mathbf{y}_k - \mathbf{y}_l\|^2)^{-1}}$$

**为什么用 t 分布而非高斯？** t 分布的尾部更重，能缓解"拥挤问题"（crowding problem）——高维空间中等距的点在低维空间中无法全部保持等距。

### 3. KL 散度损失函数

t-SNE 最小化高维分布 $P$ 和低维分布 $Q$ 之间的 KL 散度：

$$\text{KL}(P \| Q) = \sum_{i \neq j} p_{ij} \log\frac{p_{ij}}{q_{ij}}$$

**代码对应**：`tsne.kl_divergence_` 返回最终的 KL 散度值。

KL 散度是**非对称**的：
- $p_{ij}$ 大、$q_{ij}$ 小时（高维近但低维远），惩罚重 → **保持局部结构**
- $p_{ij}$ 小、$q_{ij}$ 大时（高维远但低维近），惩罚轻 → **允许全局变形**

这就是 t-SNE 擅长保持局部结构但不保持全局距离的原因。

### 4. 梯度下降优化

对低维坐标 $\mathbf{y}_i$ 的梯度为：

$$\frac{\partial\text{KL}}{\partial\mathbf{y}_i} = 4\sum_{j}(p_{ij} - q_{ij})(\mathbf{y}_i - \mathbf{y}_j)(1 + \|\mathbf{y}_i - \mathbf{y}_j\|^2)^{-1}$$

直觉：
- $p_{ij} > q_{ij}$（高维近但低维不够近）：吸引 $\mathbf{y}_i$ 向 $\mathbf{y}_j$ 移动
- $p_{ij} < q_{ij}$（高维远但低维太近）：排斥 $\mathbf{y}_i$ 远离 $\mathbf{y}_j$

### 5. PCA 初始化

**代码对应**：`init="pca"` 用 PCA 的投影结果作为 t-SNE 的初始坐标。

PCA 初始化比随机初始化的优势：
- 收敛更快（初始位置更接近最优解）
- 结果更稳定（减少随机性）
- 避免差的局部最优

### 6. 计算复杂度与加速

朴素 t-SNE 需要 $O(n^2)$ 计算所有点对的距离。加速方法：
- **Barnes-Hut 近似**：$O(n\log n)$，用四叉树/八叉树远距离点对近似为一个点
- **先用 PCA 预降维**：将高维数据降到 30-50 维后再做 t-SNE

**代码对应**：代码中先用 PCA 降到 30 维再做 t-SNE，大幅加速。

### 1. 加载数据

首先加载数据集，为后续训练和评估做准备。

In [ ]:
# Iris: 4维 → 2维（小数据集示例）
iris = load_iris()
X_iris, y_iris = iris.data, iris.target
X_iris = StandardScaler().fit_transform(X_iris)

# Digits: 64维 → 2维（中等数据集示例）
digits = load_digits()
X_digits, y_digits = digits.data, digits.target
X_digits = StandardScaler().fit_transform(X_digits)

### 2. t-SNE 降维（Iris）

运行 2. t-SNE 降维（Iris） 的代码，观察算法在该环节的行为。

In [ ]:
print("=== t-SNE 降维（Iris: 4D → 2D）===")
tsne = TSNE(n_components=2, perplexity=30, learning_rate="auto",
            init="pca", random_state=42)
X_tsne = tsne.fit_transform(X_iris)
print(f"降维后形状: {X_tsne.shape}")
# --- 输出结果 ---
print(f"KL 散度（损失）: {tsne.kl_divergence_:.4f}")

# 各类别在降维空间的中心
for c in range(3):
    mask = y_iris == c
    print(f"  类别{c} ({iris.target_names[c]}): "
          f"中心=({X_tsne[mask, 0].mean():.2f}, {X_tsne[mask, 1].mean():.2f}), "
          f"样本数={mask.sum()}")

### 3. perplexity 参数

探索关键超参数对模型行为的影响。

In [ ]:
print("\n=== perplexity 参数影响 ===")
for perp in [5, 10, 20, 30, 50]:
    tsne_p = TSNE(n_components=2, perplexity=perp, init="pca", random_state=42)
    X_p = tsne_p.fit_transform(X_iris)
    print(f"  perplexity={perp:>2}: KL散度={tsne_p.kl_divergence_:.4f}")

### 4. 不同 n_components

运行 4. 不同 n_components 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== n_components=3 ===")
tsne_3d = TSNE(n_components=3, perplexity=30, init="pca", random_state=42)
X_3d = tsne_3d.fit_transform(X_iris)
print(f"降维后形状: {X_3d.shape}")

### 5. Digits 数据集（64维 → 2维）

运行 5. Digits 数据集（64维 → 2维） 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== t-SNE 降维（Digits: 64D → 2D）===")
# 先用 PCA 降到 30 维加速（t-SNE 在高维数据上很慢）
pca = PCA(n_components=30, random_state=42)
X_digits_pca = pca.fit_transform(X_digits)
print(f"PCA 预降维: {X_digits.shape[1]}D → {X_digits_pca.shape[1]}D "
      f"(保留 {pca.explained_variance_ratio_.sum():.2%} 方差)")

tsne_d = TSNE(n_components=2, perplexity=30, init="pca", random_state=42)
X_d_tsne = tsne_d.fit_transform(X_digits_pca)
print(f"t-SNE 降维后: {X_d_tsne.shape}")
print(f"KL 散度: {tsne_d.kl_divergence_:.4f}")

### 6. t-SNE vs PCA 对比

对比不同模型或配置的性能差异。

In [ ]:
print("\n=== t-SNE vs PCA ===")
pca_2d = PCA(n_components=2)
X_pca = pca_2d.fit_transform(X_iris)
print(f"PCA  方差比: {pca_2d.explained_variance_ratio_.round(4)}, "
      f"累积: {pca_2d.explained_variance_ratio_.sum():.4f}")
# --- 输出结果 ---
print(f"t-SNE KL散度: {tsne.kl_divergence_:.4f}")

# 用 1-NN 分类器评估降维质量
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
knn = KNeighborsClassifier(n_neighbors=5)
acc_pca = cross_val_score(knn, X_pca, y_iris, cv=5).mean()
acc_tsne = cross_val_score(knn, X_tsne, y_iris, cv=5).mean()
# --- 输出结果 ---
print(f"PCA  → 1-NN CV准确率: {acc_pca:.4f}")
print(f"t-SNE → 1-NN CV准确率: {acc_tsne:.4f}")

print("\n=== t-SNE 要点 ===")
print("- 非线性降维，擅长保持高维数据的局部邻域结构")
print("- 最常用于 2D/3D 可视化，不建议用于特征工程（不可逆、不稳定）")
print("- perplexity：控制邻域大小（通常 5~50）")
print("- 学习率：'auto' 通常效果好，手动设需小心")
# --- 输出结果 ---
print("- init='pca'：用 PCA 初始化可加速收敛、提高稳定性")
print("- 计算复杂度 O(n²)，大数据集先用 PCA 预降维")
print("- 每次运行结果可能不同（受随机种子影响），不保持全局结构")

## 关键代码解释

### perplexity 参数

```python
for perp in [5, 10, 20, 30, 50]:
    tsne_p = TSNE(n_components=2, perplexity=perp, init="pca", random_state=42)
```

perplexity 可以粗略理解为"每个点关注多少个邻居"。通常 5-50，数据集大时用大值。太小 → 所有点聚成一团；太大 → 簇被挤在一起。

### PCA 预降维加速

```python
pca = PCA(n_components=30)
X_digits_pca = pca.fit_transform(X_digits)
tsne_d = TSNE(n_components=2).fit_transform(X_digits_pca)
```

t-SNE 的计算复杂度是 O(n²)，直接对 64 维数据做很慢。先用 PCA 降到 30 维可以大幅加速，同时去除噪声。

## 使用示例

```python
from sklearn.manifold import TSNE
X_2d = TSNE(n_components=2, perplexity=30, init="pca", random_state=42).fit_transform(X)
```

## 注意事项

1. **只用于可视化**：不建议用于特征工程（不可逆、不可复现、不保持全局结构）
2. **每次运行结果不同**：除非固定 random_state
3. **不保持全局结构**：只保持局部邻域关系，簇间距离没有意义
4. **大数据集慢**：O(n²)，先用 PCA 预降维
5. **不能 transform 新数据**：每次需要重新运行整个算法

## 总结与延伸

以上代码展示了 **t SNE** 的完整流程。

**解读要点**：
- 观察降维后数据的 **可分离性**——同类样本是否聚集
- 对比不同降维方法的可视化效果
- 关注 **方差解释比**（PCA）或 **邻域保持度**（UMAP）

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **openTSNE**：更快的 t-SNE 实现，支持 transform 新数据
- **FIt-SNE**：基于 FFT 加速的 t-SNE，适合百万级数据
- **t-SNE 的 perplexity 与信息论的关系**：perplexity = 2^H，H 是条件熵
